In [1]:
tabla ='qtioo10'
col_tabla = 'solopefec'
essi='essi_dat_cqx005_etl'
col_essi='sol_fec'

In [2]:
from decouple import config
from sqlalchemy import create_engine,text
import pandas as pd
from datetime import datetime, timedelta
import time 
import oracledb
import sys
import psycopg2
import numpy as np

In [3]:
inicioTiempo = time.time()
inicioProceso=time.time()
now_inicio = datetime.now()

In [4]:
#CONEXIONES
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="essi_etl"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

In [5]:
# fecha = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=13", con=connection2)
# fecha= fecha.iloc[0, 0]
# print(fecha)
now = datetime.now()
# query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=13"
# c1= text(query)
# connection2.execute(c1)

#INICIO DEL ESSI

In [6]:
fecha = '01/05/23'

In [7]:
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection0 = engine0.connect()

query0 = f"""
select
d1.INFOPEORICENASICOD,
d1.INFOPECENASICOD,
d1.INFOPESOLOPENUM,
d1.INFOPESECNUM,
d1.INFOPECPSCOD,
d1.INFOPECPSORICENASICOD,
d1.INFOPECPSCENASICOD,
d1.INFOPECPSAREHOSCOD,
d1.INFOPECPSSERVHOSCOD,

a1.solopefec as solopefec,  
a1.SOLOPEACTMEDNUM,
a1.SOLOPEBUSPACSECNUM
from {tabla} d1
left outer join qtsop10 a1 on a1.SOLOPEORICENASICOD = d1.INFOPEORICENASICOD
and a1.SOLOPECENASICOD    = d1.INFOPECENASICOD
and a1.SOLOPENUM    = d1.INFOPESOLOPENUM

where a1.{col_tabla}>='{fecha}'
"""

base1 = pd.read_sql_query(query0,con=connection0)


connection0.close()

In [8]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46615 entries, 0 to 46614
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   infopeoricenasicod     46615 non-null  object        
 1   infopecenasicod        46615 non-null  object        
 2   infopesolopenum        46615 non-null  int64         
 3   infopesecnum           46615 non-null  int64         
 4   infopecpscod           46615 non-null  object        
 5   infopecpsoricenasicod  0 non-null      object        
 6   infopecpscenasicod     0 non-null      object        
 7   infopecpsarehoscod     0 non-null      object        
 8   infopecpsservhoscod    0 non-null      object        
 9   solopefec              46615 non-null  datetime64[ns]
 10  solopeactmednum        46615 non-null  int64         
 11  solopebuspacsecnum     46615 non-null  int64         
dtypes: datetime64[ns](1), int64(4), object(7)
memory usage: 4.3+

In [9]:
base1.columns

Index(['infopeoricenasicod', 'infopecenasicod', 'infopesolopenum',
       'infopesecnum', 'infopecpscod', 'infopecpsoricenasicod',
       'infopecpscenasicod', 'infopecpsarehoscod', 'infopecpsservhoscod',
       'solopefec', 'solopeactmednum', 'solopebuspacsecnum'],
      dtype='object')

In [10]:
base1 = base1.rename(columns={
    'infopeoricenasicod': 'ori_cas',
    'infopecenasicod': 'cod_cas',
    'infopesolopenum': 'sol_num',
    'infopesecnum': 'sec_num',
    'infopecpscod': 'cod_cps',
    'infopecpsoricenasicod': 'cps_ori',
    'infopecpscenasicod': 'cps_cas',
    'infopecpsarehoscod': 'cps_are',
    'infopecpsservhoscod': 'cps_ser',
    'solopefec': 'sol_fec',
    'solopeactmednum': 'act_med',
    'solopebuspacsecnum': 'pac_sec'
})

In [11]:
base1.shape

(46615, 12)

In [12]:
base2=pd.read_sql_query(f"SELECT * FROM {essi} LIMIT 10", con=connection1)

In [13]:
base2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 17 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   ori_cas  10 non-null     object 
 1   cod_cas  10 non-null     object 
 2   des_cas  10 non-null     object 
 3   cod_red  10 non-null     object 
 4   des_red  10 non-null     object 
 5   sol_num  10 non-null     float64
 6   sec_num  10 non-null     float64
 7   cod_cps  10 non-null     object 
 8   cps_ori  0 non-null      object 
 9   cps_cas  0 non-null      object 
 10  cps_are  0 non-null      object 
 11  des_are  0 non-null      object 
 12  cps_ser  0 non-null      object 
 13  des_ser  0 non-null      object 
 14  sol_fec  10 non-null     object 
 15  act_med  10 non-null     float64
 16  pac_sec  10 non-null     float64
dtypes: float64(4), object(13)
memory usage: 1.5+ KB


#TRAEMOS TODOS LOS MAESTROS

In [14]:
cas = pd.read_sql_query(f"SELECT id_red,cod_cas,des_cas FROM dim_cas where id_red is not null", con=connection2)
cas = cas.drop_duplicates(subset=['cod_cas'])
cas=cas.dropna()
red = pd.read_sql_query(f"SELECT id_red,cod_red,des_red FROM dim_red", con=connection2)
cas_red=pd.merge(cas,red,how='left',on='id_red')
#id_red,cod_cas,des_cas,cod_red,des_red

areas = pd.read_sql_query(f"SELECT cod_are,des_are FROM dim_areas", con=connection2)
areas['cod_are']=areas['cod_are'].str.strip()

servicios = pd.read_sql_query(f"SELECT cod_ser,des_ser FROM dim_servicios", con=connection2)
servicios['cod_ser']=servicios['cod_ser'].str.strip()

In [15]:
a=base1.copy()

In [16]:
# base1=a

In [17]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46615 entries, 0 to 46614
Data columns (total 12 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   ori_cas  46615 non-null  object        
 1   cod_cas  46615 non-null  object        
 2   sol_num  46615 non-null  int64         
 3   sec_num  46615 non-null  int64         
 4   cod_cps  46615 non-null  object        
 5   cps_ori  0 non-null      object        
 6   cps_cas  0 non-null      object        
 7   cps_are  0 non-null      object        
 8   cps_ser  0 non-null      object        
 9   sol_fec  46615 non-null  datetime64[ns]
 10  act_med  46615 non-null  int64         
 11  pac_sec  46615 non-null  int64         
dtypes: datetime64[ns](1), int64(4), object(7)
memory usage: 4.3+ MB


In [18]:
base1=pd.merge(base1,cas_red,how='left',on='cod_cas')
base1=base1.drop("id_red", axis=1)
base1.shape

(46615, 15)

In [19]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 46615 entries, 0 to 46614
Data columns (total 15 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   ori_cas  46615 non-null  object        
 1   cod_cas  46615 non-null  object        
 2   sol_num  46615 non-null  int64         
 3   sec_num  46615 non-null  int64         
 4   cod_cps  46615 non-null  object        
 5   cps_ori  0 non-null      object        
 6   cps_cas  0 non-null      object        
 7   cps_are  0 non-null      object        
 8   cps_ser  0 non-null      object        
 9   sol_fec  46615 non-null  datetime64[ns]
 10  act_med  46615 non-null  int64         
 11  pac_sec  46615 non-null  int64         
 12  des_cas  46615 non-null  object        
 13  cod_red  46615 non-null  object        
 14  des_red  46615 non-null  object        
dtypes: datetime64[ns](1), int64(4), object(10)
memory usage: 5.7+ MB


In [20]:
#des_are
base1['cps_are']=base1['cps_are'].str.strip()
base1=pd.merge(base1,areas,how='left',left_on='cps_are',right_on='cod_are')
base1 = base1.drop("cod_are", axis=1)
base1.shape

(46615, 16)

In [21]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 46615 entries, 0 to 46614
Data columns (total 16 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   ori_cas  46615 non-null  object        
 1   cod_cas  46615 non-null  object        
 2   sol_num  46615 non-null  int64         
 3   sec_num  46615 non-null  int64         
 4   cod_cps  46615 non-null  object        
 5   cps_ori  0 non-null      object        
 6   cps_cas  0 non-null      object        
 7   cps_are  0 non-null      object        
 8   cps_ser  0 non-null      object        
 9   sol_fec  46615 non-null  datetime64[ns]
 10  act_med  46615 non-null  int64         
 11  pac_sec  46615 non-null  int64         
 12  des_cas  46615 non-null  object        
 13  cod_red  46615 non-null  object        
 14  des_red  46615 non-null  object        
 15  des_are  0 non-null      object        
dtypes: datetime64[ns](1), int64(4), object(11)
memory usage: 6.0+ MB


In [22]:
#des_ser
base1['cps_ser']=base1['cps_ser'].str.strip()
base1=pd.merge(base1,servicios,how='left',left_on='cps_ser',right_on='cod_ser')
base1 = base1.drop("cod_ser", axis=1)
base1.shape

(46615, 17)

In [23]:
base2.columns

Index(['ori_cas', 'cod_cas', 'des_cas', 'cod_red', 'des_red', 'sol_num',
       'sec_num', 'cod_cps', 'cps_ori', 'cps_cas', 'cps_are', 'des_are',
       'cps_ser', 'des_ser', 'sol_fec', 'act_med', 'pac_sec'],
      dtype='object')

In [24]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df2_columns - df1_columns
different_columns

set()

In [25]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df1_columns - df2_columns
different_columns

set()

In [26]:
comunes = set(base1.columns).intersection(set(base2.columns)) 
base = base1[list(comunes)]

In [27]:
base.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 46615 entries, 0 to 46614
Data columns (total 17 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   des_are  0 non-null      object        
 1   ori_cas  46615 non-null  object        
 2   sec_num  46615 non-null  int64         
 3   cod_cas  46615 non-null  object        
 4   des_red  46615 non-null  object        
 5   cps_ori  0 non-null      object        
 6   cod_cps  46615 non-null  object        
 7   sol_fec  46615 non-null  datetime64[ns]
 8   act_med  46615 non-null  int64         
 9   cps_cas  0 non-null      object        
 10  cps_are  0 non-null      object        
 11  des_ser  0 non-null      object        
 12  pac_sec  46615 non-null  int64         
 13  cps_ser  0 non-null      object        
 14  des_cas  46615 non-null  object        
 15  cod_red  46615 non-null  object        
 16  sol_num  46615 non-null  int64         
dtypes: datetime64[ns](1), int64(4),

In [28]:
base.head()

,des_are,ori_cas,sec_num,cod_cas,des_red,cps_ori,cod_cps,sol_fec,act_med,cps_cas,cps_are,des_ser,pac_sec,cps_ser,des_cas,cod_red,sol_num
0,NaN,1,1,006,RED PRESTACIONAL REBAGLIATI ...,None,58670,2029-07-10,6799154,None,None,NaN,17437280,None,H.III SUAREZ-ANGAMOS,07,5182
1,NaN,1,1,001,RED PRESTACIONAL REBAGLIATI ...,None,21552,2023-06-02,11250874,None,None,NaN,928443,None,H.N. EDGARDO REBAGLIATI MARTINS,07,63927
2,NaN,1,1,001,RED PRESTACIONAL REBAGLIATI ...,None,57500,2023-06-02,11250874,None,None,NaN,928443,None,H.N. EDGARDO REBAGLIATI MARTINS,07,63927
3,NaN,1,1,191,RED ASISTENCIAL CAJAMARCA ...,None,52332,2025-08-05,1504924,None,None,NaN,13754960,None,H.II CAJAMARCA,12,13541
4,NaN,1,1,001,RED PRESTACIONAL REBAGLIATI ...,None,38505,2023-09-02,11039911,None,None,NaN,1281848,None,H.N. EDGARDO REBAGLIATI MARTINS,07,72546


In [29]:
base.columns

Index(['des_are', 'ori_cas', 'sec_num', 'cod_cas', 'des_red', 'cps_ori',
       'cod_cps', 'sol_fec', 'act_med', 'cps_cas', 'cps_are', 'des_ser',
       'pac_sec', 'cps_ser', 'des_cas', 'cod_red', 'sol_num'],
      dtype='object')

In [30]:
borrando=f"DELETE FROM {essi} WHERE {col_essi} >='{fecha}'"
borrado = connection1.execute(borrando)

In [31]:
base.to_sql(name=f'{essi}', con=connection1, if_exists='append', index=False,chunksize=10000)

4615

In [32]:
finproceso=time.time()
tiempoproceso=finproceso - inicioProceso
tiempoproceso=round(tiempoproceso,3)
print("Proceso 1.2 completado satisfactoriamente en " + str(tiempoproceso)+" segundos")

Proceso 1.2 completado satisfactoriamente en 12.451 segundos


In [33]:
connection1.close()
connection2.close()
connection3.close()

In [34]:
engine1.dispose()
engine2.dispose()
engine3.dispose()